# Animate hybridisation terms on a grid
For a 2×2 pairs system, sweeps the frequency from 250 to 320 kHz and follows each term in its own panel: all twelve hybridisation functions R_jk (g/2π = -10, -20, -40 kHz), the sieved determinant terms of orders 2, 3 and 4, and the 3rd-order terms of the off-diagonal adjugate entry adj(**D**)ₓ₁ₓ₂ (g/2π = -20, -40 kHz). The grey disc is |z| < 1.

Runtime: about 5-30 s by default, which shows the last frame of each sweep as a static figure. Set `MAKE_ANIMATIONS = True` to write the eleven animations to `output/` (about 1 h).

# START

# Generate animations

In [ ]:
from pathlib import Path
import matplotlib
import matplotlib.animation
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
from levisim import *

set_plt_rcParams_to_xmgrace_style()

# --- run settings ---------------------------------------------------------------------------------
# False (default): every animation is stepped through its frames without being rendered and shown as
# one static figure of its last frame (with every trail), which takes seconds.
# True: every animation is also written to output/ (GIF, or MP4 when ffmpeg is installed), which
# takes minutes. output/ is not tracked by git.
MAKE_ANIMATIONS = False
# keep every n-th frame; n > 1 gives a quick test of the full run
FRAME_STRIDE = 1
# resolution of the written animation files
ANIMATION_DPI = 75
OUTPUT_DIR = Path('output')


class SweepAnimation:
    """
    A frequency-sweep animation, update(frame) over num_frames frames, rendered only on request

    show(name) writes it to OUTPUT_DIR when MAKE_ANIMATIONS is True, otherwise it just runs
    update over the frames; either way it then displays the figure, which holds the last frame.
    """
    def __init__(self, fig, update, num_frames, fps=20):
        self.fig, self.update, self.fps = fig, update, fps
        self.frames = list(range(0, num_frames, FRAME_STRIDE))
        # closed so that the notebook does not also show the empty figure when the cell ends
        plt.close(fig)

    def show(self, name):
        if MAKE_ANIMATIONS:
            OUTPUT_DIR.mkdir(exist_ok=True)
            writer = 'ffmpeg' if matplotlib.animation.writers.is_available('ffmpeg') else 'pillow'
            path = OUTPUT_DIR / f'{name}.{"mp4" if writer == "ffmpeg" else "gif"}'
            animation = matplotlib.animation.FuncAnimation(self.fig, self.update, frames=self.frames, blit=False)
            animation.save(path, writer=writer, fps=self.fps, dpi=ANIMATION_DPI)
            print(f'wrote {path} ({len(self.frames)} frames)')
        else:
            for frame in self.frames:
                self.update(frame)
        self.fig.canvas.draw()
        display(self.fig)


def generate_spectral_sweep_grid_animation(term_type, Ω_model_type, g):
    gas_kwargs = dict(P=1e-6*mbar, T=300*K, vth=500*metre/second)
    cavity_kwargs = dict(Δ=-450*kHz, L=0.96*cm, Finesse=40_000, θhom=π/4)
    toy_params = dict(Ω_model=Ω_model_type, toy_gx=g, toy_gy=g)

    system = '2×2'
    Nd = system.split('×')
    N, d = int(Nd[0]), int(Nd[1])
    D = N * d
    mtrap = MultiTrap(
        num_traps=N,
        Cartesian_axes_per_trap=d,
        ω_range=np.arange(0, 400 + 1, 1) * kHz,
        gas_kwargs=gas_kwargs,
        cavity_kwargs=cavity_kwargs,
        toy_params=toy_params,
        generate_power_series=True,
    )

    match term_type:
        case 'all':
            terms = mtrap.all_Rjk
            assert len(terms) == 12
            fig = plt.figure(layout='constrained', figsize=(3 * LATEX_FIG_WIDTH, 20), dpi=75)
            ax_dict = fig.subplot_mosaic([
                [term_string(term) for term in terms[0:3]],
                [term_string(term) for term in terms[3:6]],
                [term_string(term) for term in terms[6:9]],
                [term_string(term) for term in terms[9:12]],
            ])

        case 'determinant 2nd order':
            terms = sieve_power_series(mtrap.deranged_determinant_by_order[2], 2)
            assert len(terms) == 6
            fig = plt.figure(layout='constrained', figsize=(3 * LATEX_FIG_WIDTH, 10), dpi=75)
            ax_dict = fig.subplot_mosaic([
                [term_string(term) for term in terms[0:3]],
                [term_string(term) for term in terms[3:6]],
            ])

        case 'determinant 3rd order':
            terms = sieve_power_series(mtrap.deranged_determinant_by_order[3], 3)
            assert len(terms) == 8
            fig = plt.figure(layout='constrained', figsize=(4 * LATEX_FIG_WIDTH, 10), dpi=75)
            ax_dict = fig.subplot_mosaic([
                [term_string(term) for term in terms[0:4]],
                [term_string(term) for term in terms[4:8]],
            ])

        case 'determinant 4th order':
            terms = sieve_power_series(mtrap.deranged_determinant_by_order[4], 4)
            assert len(terms) == 9
            fig = plt.figure(layout='constrained', figsize=(3 * LATEX_FIG_WIDTH, 15), dpi=75)
            ax_dict = fig.subplot_mosaic([
                [term_string(term) for term in terms[0:3]],
                [term_string(term) for term in terms[3:6]],
                [term_string(term) for term in terms[6:9]],
            ])

        case 'x1 x2 adjugate':
            terms = mtrap.deranged_adjugate_by_order[3]['x1x2']
            assert len(terms) == 6
            fig = plt.figure(layout='constrained', figsize=(3 * LATEX_FIG_WIDTH, 10), dpi=75)
            ax_dict = fig.subplot_mosaic([
                [term_string(term) for term in terms[0:3]],
                [term_string(term) for term in terms[3:6]],
            ])


    marker = 'o'
    colours = colours_50
    lim = 2
    arrows = []
    output_trails = []
    trails = []
    for ii, ax in enumerate(list(ax_dict.values())):
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect('equal', 'box')
        ax.axhline(0, color=black, linewidth=0.5)
        ax.axvline(0, color=black, linewidth=0.5)
        ax.grid(alpha=0.1)

        circle = matplotlib.patches.Circle((0, 0), radius=1, color=black, alpha=0.05)
        ax.add_patch(circle)

        arrows.append(ax.plot([], [], color=colours[ii], marker=marker))
        output_trails.append(ax.scatter([], [], color=colours[ii], alpha=0.5, s=10))
        trails.append([])

    arrows = sum(arrows, [])

    num_terms = len(terms)
    arrows = arrows[:num_terms]
    output_trails = output_trails[:num_terms]
    trails = trails[:num_terms]
    terms = terms[:num_terms]


    ω_input = np.arange(250, 320 + 0.5, 0.5) * kHz
    def update(frame):
        ω = ω_input[frame]
        for ii, ax in enumerate(list(ax_dict.values())):
            term = terms[ii]
            fω = mtrap.hybridisation_term_value(term, ω)
            # arrows[ii].set_label(f'${term_latex(term)[1:]} = ${fω:.3f}')
            # arrows[ii].set_label(f'${term_latex(term)[1:]} = ${fω:.3f}')
            arrows[ii].set_data([0, fω.real], [0, fω.imag])
            trails[ii].append((fω.real, fω.imag))
            output_trails[ii].set_offsets(trails[ii])
            # ax.legend(loc='upper right')
            ax.set_title(f'${term_latex(term, with_ω=False)[1:]}(\omega / 2\pi$ = {ω / kHz:.1f} kHz$)$={fω:.3f}')

        return (fig, *arrows, *output_trails)

    return SweepAnimation(fig, update, len(ω_input))

ani_all_Rjk_pairs_10 = generate_spectral_sweep_grid_animation('all', 'pairs', g=-10*kHz)
ani_all_Rjk_pairs_20 = generate_spectral_sweep_grid_animation('all', 'pairs', g=-20*kHz)
ani_all_Rjk_pairs_40 = generate_spectral_sweep_grid_animation('all', 'pairs', g=-40*kHz)

ani_det_2_pairs_20 = generate_spectral_sweep_grid_animation('determinant 2nd order', 'pairs', g=-20*kHz)
ani_det_2_pairs_40 = generate_spectral_sweep_grid_animation('determinant 2nd order', 'pairs', g=-40*kHz)

ani_det_3_pairs_20 = generate_spectral_sweep_grid_animation('determinant 3rd order', 'pairs', g=-20*kHz)
ani_det_3_pairs_40 = generate_spectral_sweep_grid_animation('determinant 3rd order', 'pairs', g=-40*kHz)

ani_det_4_pairs_20 = generate_spectral_sweep_grid_animation('determinant 4th order', 'pairs', g=-20*kHz)
ani_det_4_pairs_40 = generate_spectral_sweep_grid_animation('determinant 4th order', 'pairs', g=-40*kHz)

ani_x1x2_adj_pairs_20 = generate_spectral_sweep_grid_animation('x1 x2 adjugate', 'pairs', g=-20*kHz)
ani_x1x2_adj_pairs_40 = generate_spectral_sweep_grid_animation('x1 x2 adjugate', 'pairs', g=-40*kHz)



# Play animations

## All $R_{jk}$

### g / 2π = 10 kHz

In [ ]:
ani_all_Rjk_pairs_10.show('all_Rjk_pairs_10')


### g / 2π = 20 kHz

In [ ]:
ani_all_Rjk_pairs_20.show('all_Rjk_pairs_20')


### g / 2π = 40 kHz

In [ ]:
ani_all_Rjk_pairs_40.show('all_Rjk_pairs_40')


## Determinant

### 2nd order (6 terms)

#### g / 2π = -20 kHz

In [ ]:
ani_det_2_pairs_20.show('det_2_pairs_20')


#### g / 2π = -40 kHz

In [ ]:
ani_det_2_pairs_40.show('det_2_pairs_40')


### 3rd order (8 terms)

#### g / 2π = -20 kHz

In [ ]:
ani_det_3_pairs_20.show('det_3_pairs_20')


#### g / 2π = -40 kHz

In [ ]:
ani_det_3_pairs_40.show('det_3_pairs_40')


### 4th order (9 terms)

#### g / 2π = -20 kHz

In [ ]:
ani_det_4_pairs_20.show('det_4_pairs_20')


#### g / 2π = -40 kHz

In [ ]:
ani_det_4_pairs_40.show('det_4_pairs_40')


## Adjugate

### $x_1, x_2$ off-diagonal

#### g / 2π = -20 kHz

In [ ]:
ani_x1x2_adj_pairs_20.show('x1x2_adj_pairs_20')


#### g / 2π = -40 kHz

In [ ]:
ani_x1x2_adj_pairs_40.show('x1x2_adj_pairs_40')


# END